# Dataset Builder — AI-Based Research Paper Intelligence System

## Project Context
This notebook builds the training dataset for a two-level hierarchical SciBERT classification system.
The system is part of a senior capstone project at the University of Bahrain that automatically
classifies, summarizes, extracts keywords from, and recommends similar papers for any uploaded research paper PDF.

The classifier works in two levels:
- **Level 1:** Predicts the main category (7 categories)
- **Level 2:** Predicts the subcategory (6 subcategories per main = 42 total)

---

## Final Taxonomy (Confirmed)

| Main Category | Subcategories |
|---|---|
| Computer Science | Data Structures & Algorithms, Machine Learning, Natural Language Processing, Robotics, Cryptography & Security, Artificial Intelligence |
| Mathematics | Analysis, Combinatorics, Probability, Algebra, Numerical Analysis, Optimization & Control |
| Physics | Quantum Physics, Astrophysics, Condensed Matter, High Energy Physics, Relativity & Gravity, Mathematical Physics |
| Biology & Medicine | Genomics & Genetics, Neuroscience, Epidemiology, Molecular Biology, Pharmacology, Biomedical Engineering |
| Economics & Business | Econometrics, Finance, Microeconomics, Macroeconomics, Marketing, Management |
| Engineering | Electronics & Circuits, Systems & Control, Civil & Structural Engineering, Mechanical Engineering, Chemical Engineering, Electrical Engineering |
| Chemistry | Organic Chemistry, Inorganic Chemistry, Physical Chemistry, Analytical Chemistry, Computational Chemistry, Biochemistry |

**7 main categories × 6 subcategories = 42 subcategories total**
**Target: 20,000 papers per subcategory = 840,000 papers total**

---

## Key Design Decisions and Why We Made Them

### Why we redesigned the taxonomy from scratch
The original project proposal used 8 arXiv-aligned categories (CS, Math, Physics, Statistics,
EESS, Economics, Quantitative Biology, Quantitative Finance). After examining the actual arXiv
dataset distribution we found severe imbalances:
- CS alone had 466,560 papers
- Quantitative Finance had only 8,459 papers
- The ratio between largest and smallest was 55:1

A model trained on this distribution would be heavily biased toward predicting CS and Physics
for almost everything. We redesigned the taxonomy from scratch based on what the data actually supports.

### Why we chose 7 main categories
We started by filtering all arXiv subcategory codes to those with 20,000+ papers. This gave us
a clear picture of what domains are well represented. We then made academic decisions on top:
- CS, Mathematics, Physics are naturally well supported by arXiv
- Biology & Medicine, Economics & Business, Engineering, Chemistry are important academic domains
  but underrepresented in arXiv — so we supplement with PubMed and S2ORC
- We dropped the original Economics, Q-Finance, and Q-Bio categories because arXiv simply
  doesn't have enough papers in those domains to train reliably
- We avoided combining unrelated domains into one category (e.g. Statistics + Engineering
  was considered and rejected because it's academically unnatural)

### Why we use multiple data sources
arXiv is heavily biased toward Physics, Math, and CS because those communities adopted it earliest.
To get balanced coverage across all 7 main categories we use:
- **arXiv** — CS, Mathematics, Physics, Systems & Control (eess.SY), Physical Chemistry (physics.chem-ph)
- **PubMed** — Biology & Medicine (6 subcategories)
- **S2ORC (Semantic Scholar Open Research Corpus)** — Economics & Business, Engineering, Chemistry (remaining subcategories)
- **ResearchGate scrape** — UoB CS/IT professor papers (added on top, slot into CS subcategories)

### Why we cap at 20,000 papers per subcategory
SciBERT is already pretrained on 3.1 million scientific papers. Fine-tuning doesn't train
from scratch — it adjusts existing weights for our specific task. The performance sweet spot
for fine-tuning BERT-based models is 10,000–20,000 papers per class:
- Below 5,000: acceptable but may struggle on edge cases
- 5,000–10,000: good performance
- 10,000–20,000: excellent performance
- Above 20,000: diminishing returns, just increases training time

We chose 20,000 (top of the sweet spot) for maximum accuracy while keeping training
feasible within a single Kaggle T4 GPU session (12 hours).

### Why we cap at the subcategory level not the main category level
Capping at the subcategory level automatically balances both levels:
- Level 1 balance: each main category has 6 subcategories × 20K = 120K papers
- Level 2 balance: each subcategory has exactly 20K papers
This gives us perfect balance at both levels simultaneously.

### Why we use title + abstract as model input (not full text)
- SciBERT has a maximum input of 512 tokens
- Full paper text is tens of thousands of words — far too long
- Abstracts are 150–300 words — fits perfectly within 512 tokens
- Title + abstract captures the core contribution of a paper
- Using full text would require chunking strategies that add complexity
- Short sequences also mean much faster training time on Kaggle

### Why we take the first listed arXiv category as primary
arXiv papers can have multiple categories (e.g. "cs.LG cs.AI stat.ML"). We take only
the first one as the primary category because:
- The author lists categories in order of relevance — first is most relevant
- Using multiple categories per paper would cause label ambiguity
- It keeps the preprocessing simple and consistent

### Why we map multiple arXiv codes to single subcategory labels (Physics)
arXiv split some broad categories into subcodes over time. For example:
- The old `astro-ph` code was later split into astro-ph.GA, astro-ph.SR, astro-ph.HE, etc.
- Papers from different eras use different codes but are all Astrophysics
- We map all variants to the same label and pool them together before capping
- Same approach for Condensed Matter (cond-mat.*) and High Energy Physics (hep-th + hep-ph)

### Why Physical Chemistry is below 20K from arXiv
arXiv's physics.chem-ph code only has 14,139 papers — chemistry as a field historically
publishes in dedicated journals rather than arXiv. We use all 14,139 arXiv papers and
will top up the remaining ~5,861 from S2ORC when we process the Chemistry section.

---

## Data Sources

### Source 1: arXiv (~394K papers after sampling)
- **File:** arxiv-metadata-oai-snapshot.json (~3M papers, JSONL format)
- **Download:** Kaggle — Cornell University arXiv dataset
- **Coverage:** CS (6 subs), Mathematics (6 subs), Physics (6 subs),
  Systems & Control (Engineering), Physical Chemistry (Chemistry)
- **Status:** ✅ COMPLETE

### Source 2: PubMed (target: 120K papers)
- **File:** pubmed25n000X.xml.gz (~30K papers per file, XML format)
- **Download:** https://ftp.ncbi.nlm.nih.gov/pubmed/baseline/
- **Coverage:** Biology & Medicine (6 subcategories)
- **Filtering method:** MeSH terms (Medical Subject Headings)
- **Status:** 🔄 IN PROGRESS — downloading files

### Source 3: S2ORC — Semantic Scholar Open Research Corpus (target: ~320K papers)
- **Coverage:** Economics & Business (6 subs), Engineering minus Systems & Control (5 subs),
  Chemistry minus Physical Chemistry (5 subs) + top up Physical Chemistry
- **Filtering method:** fieldsOfStudy tags
- **Status:** ⏳ TODO

### Source 4: ResearchGate scrape (small, bonus)
- **Coverage:** UoB CS/IT professor papers — slot into existing CS subcategories
- **Method:** BeautifulSoup scraper on professor profile URLs
- **Status:** ⏳ TODO — professor URLs not yet collected

---

## Unified Data Format
Every paper from every source is converted to the same 5-column format:

| Column | Description | Example |
|---|---|---|
| title | Paper title, newlines removed | "Deep Residual Learning for Image Recognition" |
| abstract | Paper abstract, newlines removed | "We present a residual learning framework..." |
| main_label | Main category name | "Computer Science" |
| sub_label | Subcategory name | "Machine Learning" |
| source | Where the paper came from | "arxiv" / "pubmed" / "s2orc" / "uob" |

The `source` column is for our reference only — the model never sees it during training.

---

## Processing Pipeline
arXiv JSON     →  extract title + abstract  →  map category codes  →  cap at 20K  →  dataset_arxiv.csv
PubMed XML     →  extract title + abstract  →  filter MeSH terms   →  cap at 20K  →  dataset_pubmed.csv
S2ORC JSON     →  extract title + abstract  →  filter field tags   →  cap at 20K  →  dataset_s2orc.csv
ResearchGate   →  scrape title + abstract   →  manual labels       →  add all     →  dataset_uob.csv
↓
merge_all.csv (final)

---

## Current Progress

| Step | Status |
|---|---|
| Taxonomy design | ✅ Complete |
| arXiv download | ✅ Complete |
| arXiv preprocessing | ✅ Complete — 394,139 papers saved to dataset_arxiv.csv |
| PubMed download | 🔄 In progress |
| PubMed preprocessing | ⏳ Todo |
| S2ORC download | ⏳ Todo |
| S2ORC preprocessing | ⏳ Todo |
| ResearchGate scrape | ⏳ Todo — URLs not yet collected |
| Final merge | ⏳ Todo |

---

## After This Notebook — What Happens Next
Once the final merged CSV is ready it gets uploaded to Kaggle as a dataset.
Two separate training notebooks will then fine-tune SciBERT:
- **Training Notebook 1:** Level 1 classifier — 7 main categories
- **Training Notebook 2:** Level 2 classifier — 42 subcategories
  (runs with category name concatenated to input as a hint)

The trained model weights get downloaded and placed in:
- `backend/app/services/classifier.py` — replacing the current hardcoded stub

In [15]:
import json
import pandas as pd
import random
from collections import defaultdict, Counter
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

random.seed(42)


In [16]:
# CONFIGURATION
FILE_PATH = r"C:\Users\hp\Desktop\arxiv-metadata-oai-snapshot.json"
CAP_PER_SUB = 20000

In [17]:
# PAPER STRUCTURE
# Open the dataset, find the first valid paper, and print its content.

print("Finding the first usable paper...\n")

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        paper = json.loads(line)

        # Ensure it has the minimum we care about
        if paper.get('title') and paper.get('categories'):
            print("Keys present in a paper:")
            for key in paper:
                print(f"  • {key}")
            
            print("\nSample values (truncated for long fields):")
            for key, value in paper.items():
                if isinstance(value, str) and len(value) > 150:
                    value = value[:150] + "..."
                print(f"  {key}: {value}")
            
            # Only need one example
            break

Finding the first usable paper...

Keys present in a paper:
  • id
  • submitter
  • authors
  • title
  • comments
  • journal-ref
  • doi
  • report-no
  • categories
  • license
  • abstract
  • versions
  • update_date
  • authors_parsed

Sample values (truncated for long fields):
  id: 0704.0001
  submitter: Pavel Nadolsky
  authors: C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan
  title: Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies
  comments: 37 pages, 15 figures; published version
  journal-ref: Phys.Rev.D76:013009,2007
  doi: 10.1103/PhysRevD.76.013009
  report-no: ANL-HEP-PR-07-12
  categories: hep-ph
  license: None
  abstract:   A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders....
  versions: [{'version': 'v1', 'created': 'Mon, 2 Apr 2007 19:18:42 GMT'}, {'version': 'v2', 'created': 'Tue, 24 Jul 2007 20:10:27 GMT'}]
  update_da

In [18]:
# ALL AVAILABLE CATEGORIES
# Gather all unique main categories (e.g., cs, math, quant-ph)
# and all unique subcategories (e.g., cs.CV, astro-ph.GA).

main_categories = set()
sub_categories = set()

print("Scanning all papers for categories...")

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        paper = json.loads(line)
        cats_raw = paper.get('categories')
        if not cats_raw:
            continue
        
        # Primary category = first in the list
        primary = cats_raw.split()[0]
        
        # Main category: everything before the dot, or the whole string if no dot
        main = primary.split('.')[0] if '.' in primary else primary
        
        main_categories.add(main)
        sub_categories.add(primary)

# Print sorted lists for easy scanning
print("\nAll unique MAIN categories (arXiv codes):")
for cat in sorted(main_categories):
    print(f"  {cat}")

print(f"\nAll unique SUBCATEGORIES (full codes):")
for cat in sorted(sub_categories):
    print(f"  {cat}")

print(f"\nTotal unique main categories: {len(main_categories)}")
print(f"Total unique subcategories:   {len(sub_categories)}")

Scanning all papers for categories...

All unique MAIN categories (arXiv codes):
  acc-phys
  adap-org
  alg-geom
  ao-sci
  astro-ph
  atom-ph
  bayes-an
  chao-dyn
  chem-ph
  cmp-lg
  comp-gas
  cond-mat
  cs
  dg-ga
  econ
  eess
  funct-an
  gr-qc
  hep-ex
  hep-lat
  hep-ph
  hep-th
  math
  math-ph
  mtrl-th
  nlin
  nucl-ex
  nucl-th
  patt-sol
  physics
  plasm-ph
  q-alg
  q-bio
  q-fin
  quant-ph
  solv-int
  stat
  supr-con

All unique SUBCATEGORIES (full codes):
  acc-phys
  adap-org
  alg-geom
  ao-sci
  astro-ph
  astro-ph.CO
  astro-ph.EP
  astro-ph.GA
  astro-ph.HE
  astro-ph.IM
  astro-ph.SR
  atom-ph
  bayes-an
  chao-dyn
  chem-ph
  cmp-lg
  comp-gas
  cond-mat
  cond-mat.dis-nn
  cond-mat.mes-hall
  cond-mat.mtrl-sci
  cond-mat.other
  cond-mat.quant-gas
  cond-mat.soft
  cond-mat.stat-mech
  cond-mat.str-el
  cond-mat.supr-con
  cs.AI
  cs.AR
  cs.CC
  cs.CE
  cs.CG
  cs.CL
  cs.CR
  cs.CV
  cs.CY
  cs.DB
  cs.DC
  cs.DL
  cs.DM
  cs.DS
  cs.ET
  cs.FL
  cs.GL
  c

In [19]:
# CATEGORY DISTRIBUTION & DATA QUALITY
# Counts papers per main category and subcategory, counts missing
# fields, and reports summary statistics.

main_counts = Counter()
sub_counts = Counter()

total_papers = 0
missing_abstract = 0
missing_title = 0
missing_cats = 0
multi_cat = 0

print("Processing full dataset for counts... (may take a minute)")

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        paper = json.loads(line)
        total_papers += 1
        
        # Data quality checks
        if not paper.get('abstract'):
            missing_abstract += 1
        if not paper.get('title'):
            missing_title += 1
        
        cats_raw = paper.get('categories')
        if not cats_raw:
            missing_cats += 1
            continue
        
        cats = cats_raw.split()
        if len(cats) > 1:
            multi_cat += 1
        
        primary = cats[0]
        main = primary.split('.')[0] if '.' in primary else primary
        
        main_counts[main] += 1
        sub_counts[primary] += 1

# ─── Print summary ───
print(f"\n{'='*50}")
print(f"Total papers:               {total_papers:>10,}")
print(f"Missing abstracts:          {missing_abstract:>10,}")
print(f"Missing titles:             {missing_title:>10,}")
print(f"Missing category field:     {missing_cats:>10,}")
print(f"Papers with >1 category:    {multi_cat:>10,}")
print(f"Unique main categories:     {len(main_counts):>10,}")
print(f"Unique subcategories:       {len(sub_counts):>10,}")
print(f"{'='*50}")

# All main categories sorted by count
print("\nAll main categories by paper count:")
for cat, count in main_counts.most_common():
    print(f"  {cat:<20} {count:>10,}")

# Top 40 subcategories sorted by count
print("\nTop 40 subcategories by paper count:")
for cat, count in sub_counts.most_common(40):
    print(f"  {cat:<30} {count:>10,}")

Processing full dataset for counts... (may take a minute)

Total papers:                3,021,763
Missing abstracts:                   0
Missing titles:                      0
Missing category field:              0
Papers with >1 category:     1,443,795
Unique main categories:             38
Unique subcategories:              172

All main categories by paper count:
  cs                      752,557
  math                    575,967
  cond-mat                346,986
  astro-ph                334,097
  physics                 209,275
  hep-ph                  142,225
  quant-ph                129,940
  hep-th                  112,595
  eess                     71,985
  gr-qc                    71,179
  stat                     61,470
  nucl-th                  35,565
  math-ph                  34,044
  q-bio                    33,938
  hep-ex                   24,979
  nlin                     20,701
  hep-lat                  19,073
  q-fin                    13,404
  nucl-ex          

In [20]:
# ─── CATEGORY MAP ─────────────────────────────────────────────────
# Maps arXiv primary category codes to our taxonomy
# For Physics subcategories with multiple codes (Astrophysics, Condensed Matter,
# High Energy), all codes map to the same subcategory label — they get pooled
# together then capped at 20K

ARXIV_CATEGORY_MAP = {
    # Computer Science
    "cs.DS": {"main": "Computer Science", "sub": "Data Structures & Algorithms"},
    "cs.LG": {"main": "Computer Science", "sub": "Machine Learning"},
    "cs.CL": {"main": "Computer Science", "sub": "Natural Language Processing"},
    "cs.RO": {"main": "Computer Science", "sub": "Robotics"},
    "cs.CR": {"main": "Computer Science", "sub": "Cryptography & Security"},
    "cs.AI": {"main": "Computer Science", "sub": "Artificial Intelligence"},

    # Mathematics
    "math.AP": {"main": "Mathematics", "sub": "Analysis"},
    "math.CO": {"main": "Mathematics", "sub": "Combinatorics"},
    "math.PR": {"main": "Mathematics", "sub": "Probability"},
    "math.AG": {"main": "Mathematics", "sub": "Algebra"},
    "math.NA": {"main": "Mathematics", "sub": "Numerical Analysis"},
    "math.OC": {"main": "Mathematics", "sub": "Optimization & Control"},

    # Physics — single codes
    "quant-ph": {"main": "Physics", "sub": "Quantum Physics"},
    "gr-qc":    {"main": "Physics", "sub": "Relativity & Gravity"},
    "math-ph":  {"main": "Physics", "sub": "Mathematical Physics"},

    # Physics — Astrophysics
    "astro-ph":    {"main": "Physics", "sub": "Astrophysics"},
    "astro-ph.GA": {"main": "Physics", "sub": "Astrophysics"},
    "astro-ph.SR": {"main": "Physics", "sub": "Astrophysics"},
    "astro-ph.HE": {"main": "Physics", "sub": "Astrophysics"},
    "astro-ph.CO": {"main": "Physics", "sub": "Astrophysics"},
    "astro-ph.EP": {"main": "Physics", "sub": "Astrophysics"},
    "astro-ph.IM": {"main": "Physics", "sub": "Astrophysics"},

    # Physics — Condensed Matter (added dis-nn and other)
    "cond-mat":           {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.mtrl-sci":  {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.mes-hall":  {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.str-el":    {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.stat-mech": {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.supr-con":  {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.soft":      {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.quant-gas": {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.dis-nn":    {"main": "Physics", "sub": "Condensed Matter"},
    "cond-mat.other":     {"main": "Physics", "sub": "Condensed Matter"},

    # Physics — High Energy
    "hep-th": {"main": "Physics", "sub": "High Energy Physics"},
    "hep-ph": {"main": "Physics", "sub": "High Energy Physics"},

    # Engineering
    "eess.SY": {"main": "Engineering", "sub": "Systems & Control"},

    # Chemistry
    "physics.chem-ph": {"main": "Chemistry", "sub": "Physical Chemistry"},
}

In [21]:
def clean_field(text: str) -> str:
    # Remove display math $$...$$
    text = re.sub(r'\$\$[^$]*\$\$', '', text)
    
    # Remove inline math $...$
    text = re.sub(r'\$[^$]*\$', '', text)
    
    # Remove LaTeX commands with optional arguments: \section[short]{Long}
    text = re.sub(r'\\[a-zA-Z]+\[[^\]]*\]\{[^}]*\}', '', text)
    
    # Remove LaTeX commands with required argument: \textbf{...}
    text = re.sub(r'\\[a-zA-Z]+\{[^}]*\}', '', text)
    
    # Remove residual LaTeX commands without arguments: \smallskip
    text = re.sub(r'\\[a-zA-Z]+', '', text)
    
    # Replace newlines with space
    text = text.replace('\n', ' ')
    
    # Collapse multiple spaces into one
    text = re.sub(r' +', ' ', text)
    
    return text.strip()

In [22]:
# PASS 1: COLLECT ALL VALID PAPERS GROUPED BY SUBCATEGORY
papers_by_sub = defaultdict(list)
skipped_not_in_taxonomy = 0
skipped_missing_fields = 0

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        paper = json.loads(line)

        # take only the first listed category as primary
        primary_category = paper['categories'].split()[0]

        # skip if not in our taxonomy
        if primary_category not in ARXIV_CATEGORY_MAP:
            skipped_not_in_taxonomy += 1
            continue

        # skip if title or abstract is missing
        if not paper.get('abstract') or not paper.get('title'):
            skipped_missing_fields += 1
            continue

        mapping = ARXIV_CATEGORY_MAP[primary_category]

        papers_by_sub[mapping['sub']].append({
            'title':      clean_field(paper['title']),
            'abstract':   clean_field(paper['abstract']),
            'main_label': mapping['main'],
            'sub_label':  mapping['sub'],
            'source':     'arxiv'
        })

print("Papers collected per subcategory (before cap):")
for sub, papers in sorted(papers_by_sub.items()):
    print(f"  {sub}: {len(papers):,}")
print(f"\nSkipped (not in taxonomy): {skipped_not_in_taxonomy:,}")
print(f"Skipped (missing title/abstract): {skipped_missing_fields:,}")

Papers collected per subcategory (before cap):
  Algebra: 39,905
  Analysis: 57,207
  Artificial Intelligence: 37,310
  Astrophysics: 334,097
  Combinatorics: 52,773
  Condensed Matter: 346,986
  Cryptography & Security: 32,209
  Data Structures & Algorithms: 18,005
  High Energy Physics: 254,820
  Machine Learning: 131,040
  Mathematical Physics: 34,044
  Natural Language Processing: 79,999
  Numerical Analysis: 34,795
  Optimization & Control: 38,674
  Physical Chemistry: 14,139
  Probability: 43,747
  Quantum Physics: 129,940
  Relativity & Gravity: 71,179
  Robotics: 38,569
  Systems & Control: 21,327

Skipped (not in taxonomy): 1,210,998
Skipped (missing title/abstract): 0


In [23]:
# PASS 2: SAMPLE UP TO CAP_PER_SUB PER SUBCATEGORY
arxiv_papers = []

for sub, papers in papers_by_sub.items():
    if len(papers) > CAP_PER_SUB:
        sampled = random.sample(papers, CAP_PER_SUB)
    else:
        sampled = papers
        print(f"  WARNING: {sub} only has {len(papers):,} papers — using all")
    arxiv_papers.extend(sampled)

random.shuffle(arxiv_papers)
df_arxiv = pd.DataFrame(arxiv_papers)

print(f"\nTotal arXiv papers after sampling: {len(df_arxiv):,}")
print("\nFinal distribution:")
print(df_arxiv['sub_label'].value_counts())


Total arXiv papers after sampling: 392,144

Final distribution:
sub_label
Astrophysics                    20000
Numerical Analysis              20000
Optimization & Control          20000
Combinatorics                   20000
Analysis                        20000
Probability                     20000
High Energy Physics             20000
Algebra                         20000
Systems & Control               20000
Condensed Matter                20000
Cryptography & Security         20000
Robotics                        20000
Relativity & Gravity            20000
Quantum Physics                 20000
Mathematical Physics            20000
Machine Learning                20000
Artificial Intelligence         20000
Natural Language Processing     20000
Data Structures & Algorithms    18005
Physical Chemistry              14139
Name: count, dtype: int64


In [24]:
# SAVE ARXIV SECTION
ARXIV_OUTPUT = r"C:\Users\hp\Desktop\Research-System\training\dataset_arxiv.csv"
df_arxiv.to_csv(ARXIV_OUTPUT, index=False)
print(f"Saved to {ARXIV_OUTPUT}")

Saved to C:\Users\hp\Desktop\Research-System\training\dataset_arxiv.csv


In [25]:
# POST-RUN AUDIT: CHECK FOR MISSING CATEGORY VARIANTS

# get all prefixes we care about from the map
# e.g. "astro-ph.GA" → "astro-ph", "cs.LG" → "cs", "quant-ph" → "quant-ph"
prefixes_in_map = set()
for code in ARXIV_CATEGORY_MAP:
    prefix = code.split('.')[0] if '.' in code else code
    prefixes_in_map.add(prefix)

# now scan raw file and find ALL codes that share those prefixes
variants_found = Counter()

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        paper = json.loads(line)
        primary = paper['categories'].split()[0]
        prefix = primary.split('.')[0] if '.' in primary else primary
        if prefix in prefixes_in_map:
            variants_found[primary] += 1

print("All variants found for prefixes in your taxonomy:")
print(f"{'Code':<30} {'Count':>10}  {'In Map?'}")
print("-" * 55)
for cat, count in sorted(variants_found.items()):
    in_map = "✅" if cat in ARXIV_CATEGORY_MAP else "❌ MISSING"
    print(f"  {cat:<30} {count:>8,}  {in_map}")

All variants found for prefixes in your taxonomy:
Code                                Count  In Map?
-------------------------------------------------------
  astro-ph                         94,246  ✅
  astro-ph.CO                      44,635  ✅
  astro-ph.EP                      26,197  ✅
  astro-ph.GA                      54,365  ✅
  astro-ph.HE                      45,748  ✅
  astro-ph.IM                      20,871  ✅
  astro-ph.SR                      48,035  ✅
  cond-mat                         11,357  ✅
  cond-mat.dis-nn                  12,408  ✅
  cond-mat.mes-hall                69,109  ✅
  cond-mat.mtrl-sci                70,368  ✅
  cond-mat.other                    6,998  ✅
  cond-mat.quant-gas               15,142  ✅
  cond-mat.soft                    31,650  ✅
  cond-mat.stat-mech               43,553  ✅
  cond-mat.str-el                  52,458  ✅
  cond-mat.supr-con                33,943  ✅
  cs.AI                            37,310  ✅
  cs.AR                          